## Simulation of DT Model - (Dual Trap: Shallow Trap with De-Trapping, Deep Non-Radiative Trap)

In [ ]:
from globalfit_functions import *
import matplotlib.pyplot as plt

#### Define Simulation Parameters

In [ ]:
#Time between 0 and 100 microseconds, logarithmically spaced
time = (jnp.logspace(0, np.log10(100001), 1000))-1

#Initial Carrier Densities
n0s = [1e14, 1e15, 1e16, 1e17, 1e18]

#Rate Constants
k_c = 1e-3 #Shallow Trap Capture (ns^-1)
k_deep = 1e-4 #Deep Trap Capture (ns^-1)
k_e = 1e-4 #Shallow Trap Emission / De-Trapping (ns^-1)
k_rad = 1e-19 #Radiative (cm^3 ns^-1)
k_aug = 1e-37 #Auger (cm^6 ns^-1)
p0 = 1e13 #Background Hole Density
bkg = 1e-5 #Background Signal

time_label = 'Time (ns)'
qfls_label = 'QFLS (eV)'
PL_label = 'PL (a.u.)'
diff_decay_label = fr'$\tau_{{diff}}$ (s)'
diff_const_label = fr'$k_{{diff}}$ ($cm^{{3}}s^{{-1}})$'


#### Simulate TRPL Decays and Differential Transformations

In [ ]:
fig, ax = plt.subplots(1,3 , figsize=(15,5))
ax_inset_1 = ax[1].inset_axes([0.55, 0.55, 0.425, 0.425])
ax_inset_2 = ax[2].inset_axes([0.55, 0.55, 0.425, 0.425])
for i, n in enumerate(n0s):
    print(f'Simulating for n0 = {n:.2e}')
    color = colorFader('red', 'blue', factor=float(i)/len(n0s))

    trpl, n_e, n_p, n_t = TRPL_DT(time, n, k_c, k_deep, k_e, k_rad, k_aug, p0, 0)
    trpl_with_bkg = TRPL_DT(time, n, k_c, k_deep, k_e, k_rad, k_aug, p0, bkg)[0]
    trpl = 10**trpl
    trpl_with_bkg = 10**trpl_with_bkg
    ax[0].loglog(time, trpl_with_bkg/trpl_with_bkg[0], color=color, alpha=0.4, linestyle='dashed')
    ax[0].loglog(time, trpl/trpl[0], color=color, label =f'10$^{{{int(np.log10(n))}}}$')

    tau = diff_lifetime(time, trpl)
    d = diff_constant(time, trpl, n)
    qfls = relative_QFLS(trpl, n0=n, n0_max=n0s[0], eg=1.55)

    ax[1].semilogy(qfls, tau, color=color)
    ax_inset_1.loglog(time, tau, color=color)
    ax[2].semilogy(qfls, d, color=color)
    ax_inset_2.loglog(time, d, color=color)

ax[0].set_xlabel(time_label, fontsize='large')
ax[0].set_ylabel(PL_label, fontsize='large')
ax[0].set_title('TRPL Decay Curves')
ax[0].legend(title=fr'$\Delta$n$_0$ (cm$^{{-3}}$)', loc = 'lower left', fontsize='large', title_fontsize='large')
ax[1].set_xlabel(qfls_label, fontsize='large')
ax[1].set_ylabel(diff_decay_label, fontsize='large')
ax[1].set_title('Diff. Lifetime vs QFLS', fontsize='large')
ax_inset_1.set_xlabel(time_label, fontsize='medium')
ax_inset_2.set_xlabel(time_label, fontsize='medium')
ax[2].set_xlabel(qfls_label, fontsize='large')
ax[2].set_ylabel(diff_const_label, fontsize='large')
ax[2].set_title('Diff. Constant vs QFLS', fontsize='large')
ax[0].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[1].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[2].tick_params(axis='both', which='both', labelsize='large', direction='in')

fig.tight_layout()

#### Simulate Charge Carriers

In [ ]:
fig, ax = plt.subplots(1,3 , figsize=(15,5))
for i, n in enumerate(n0s):
    if i % 2 == 0:
        print(f'Simulating for n0 = {n:.2e}')

        trpl, n_e, n_p, n_t = TRPL_DT(time, n, k_c, k_deep, k_e, k_rad, k_aug, p0, 0)

        ax[i//2].loglog(time, n_e, color='black', linestyle='solid')
        ax[i//2].loglog(time, n_t, color='red', linestyle='dashed')
        ax[i//2].loglog(time, n_p, color='blue', linestyle='dashdot')
        ax[i//2].set_xlabel(time_label, fontsize='x-large')
        ax[i//2].set_ylabel(r'Carrier Density (cm$^{-3}$)', fontsize='x-large')
        ax[i//2].set_title(f'Carrier Densities for $\Delta$n$_0$ = 10$^{{{int(np.log10(n))}}}$ cm$^{{-3}}$', fontsize='x-large')
        ax[i//2].set_ylim(1e11,1.5e18)
        ax[i//2].tick_params(axis='both', which='major', labelsize='x-large', direction='in')
fig.legend(['Free Electrons', 'Shallow-Trapped Electrons', 'Holes'], fontsize='x-large', ncols=3, loc='upper center', bbox_to_anchor=(0.5, -0.01))
fig.tight_layout()

#### Contributions of Each Physical Process

In [ ]:
planner2_colors = ['#AED6F1', '#4A235A', '#2E86C1', '#17A589', '#FFDAB9']
fig, ax = plt.subplots(1,3 , figsize=(15,5))
for i, n in enumerate(n0s):
    if i % 2 == 0:
        print(f'Simulating for n0 = {n:.2e}')

        trpl, n_e, n_p, n_t = TRPL_DT(time, n, k_c, k_deep, k_e, k_rad, k_aug, p0, 0)
        radiative = k_rad * n_e * n_p
        auger = 0.5 * k_aug * (n_e**2 * n_p + n_p**2 * n_e)
        capture = k_c * n_e
        emission = k_e * n_t
        deep = k_deep * n_e

        total = radiative + auger + capture + emission + deep

        radiative_norm, auger_norm, capture_norm, emission_norm, deep_norm = [100*x/total for x in [radiative, auger, capture, emission, deep]]

        ax[i//2].stackplot(time, radiative_norm, auger_norm, capture_norm, emission_norm, deep_norm, colors=planner2_colors, labels=['Radiative', 'Auger', 'Shallow Capture', 'De-Trapping', 'Deep Trapping'])
        ax[i//2].set_xlabel(time_label, fontsize='x-large')
        ax[i//2].set_ylabel('Contribution to Decay (%)', fontsize='x-large')
        ax[i//2].set_title(f'Decay Contributions for $\Delta$n$_0$ = 10$^{{{int(np.log10(n))}}}$ cm$^{{-3}}$', fontsize='x-large')
        ax[i//2].set_ylim(0,100)
        ax[i//2].set_xscale('log')
        ax[i//2].set_xlim(0,1e5)
        ax[i//2].tick_params(axis='both', which='major', labelsize='x-large', direction='in')
fig.legend(['Radiative', 'Auger', 'Shallow Capture', 'De-Trapping', 'Deep Trapping'], fontsize='x-large', ncols=5, loc='upper center', bbox_to_anchor=(0.5, -0.01))
fig.tight_layout()

#### Effect of k_deep

In [ ]:
k_deeps = [1e-3, 1e-4, 1e-5]
n0 = 1e16
fig, ax = plt.subplots(1,3 , figsize=(15,5))
ax_inset_1 = ax[1].inset_axes([0.15, 0.15, 0.425, 0.425])
ax_inset_2 = ax[2].inset_axes([0.55, 0.55, 0.425, 0.425])
for i, k_deep in enumerate(k_deeps):
    print(f'Simulating for k_deep = {k_deep:.2e}')
    color = colorFader('red', 'blue', factor=float(i)/len(k_deeps))

    trpl, n_e, n_p, n_t = TRPL_DT(time, n0, k_c, k_deep, k_e, k_rad, k_aug, p0, 0)
    trpl_with_bkg = TRPL_DT(time, n0, k_c, k_deep, k_e, k_rad, k_aug, p0, bkg)[0]
    trpl = 10**trpl
    trpl_with_bkg = 10**trpl_with_bkg
    ax[0].loglog(time, trpl_with_bkg/trpl_with_bkg[0], color=color, alpha=0.4, linestyle='dashed')
    ax[0].loglog(time, trpl/trpl[0], color=color, label =f'10$^{{{int(np.log10(k_deep))}}}$')

    tau = diff_lifetime(time, trpl)
    d = diff_constant(time, trpl, n0)
    qfls = relative_QFLS(trpl, n0=n0, n0_max=1e18, eg=1.55)

    ax[1].semilogy(qfls, tau, color=color)
    ax_inset_1.loglog(time, tau, color=color)
    ax[2].semilogy(qfls, d, color=color)
    ax_inset_2.loglog(time, d, color=color)

ax[0].set_xlabel(time_label, fontsize='large')
ax[0].set_ylabel(PL_label, fontsize='large')
ax[0].set_title('TRPL Decay Curves')
ax[0].legend(title=fr'k$_{{deep}}$ (ns$^{{-1}}$)', loc = 'lower left', fontsize='large', title_fontsize='large')
ax[1].set_xlabel(qfls_label, fontsize='large')
ax[1].set_ylabel(diff_decay_label, fontsize='large')
ax[1].set_title('Diff. Lifetime vs QFLS', fontsize='large')
ax_inset_1.set_xlabel(time_label, fontsize='medium')
ax_inset_2.set_xlabel(time_label, fontsize='medium')
ax[2].set_xlabel(qfls_label, fontsize='large')
ax[2].set_ylabel(diff_const_label, fontsize='large')
ax[2].set_title('Diff. Constant vs QFLS', fontsize='large')
ax[0].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[1].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[2].tick_params(axis='both', which='both', labelsize='large', direction='in')

fig.tight_layout()
k_deep = 1e-4 #Reset to the default value

#### Effect of k_e

In [ ]:
k_es = [1e-3, 1e-4, 1e-5]
n0 = 1e16
fig, ax = plt.subplots(1,3 , figsize=(15,5))
ax_inset_1 = ax[1].inset_axes([0.15, 0.15, 0.425, 0.425])
ax_inset_2 = ax[2].inset_axes([0.55, 0.55, 0.425, 0.425])
for i, k_e in enumerate(k_es):
    print(f'Simulating for k_e = {k_e:.2e}')
    color = colorFader('red', 'blue', factor=float(i)/len(k_es))

    trpl, n_e, n_p, n_t = TRPL_DT(time, n0, k_c, k_deep, k_e, k_rad, k_aug, p0, 0)
    trpl_with_bkg = TRPL_DT(time, n0, k_c, k_deep, k_e, k_rad, k_aug, p0, bkg)[0]
    trpl = 10**trpl
    trpl_with_bkg = 10**trpl_with_bkg
    ax[0].loglog(time, trpl_with_bkg/trpl_with_bkg[0], color=color, alpha=0.4, linestyle='dashed')
    ax[0].loglog(time, trpl/trpl[0], color=color, label =f'10$^{{{int(np.log10(k_e))}}}$')

    tau = diff_lifetime(time, trpl)
    d = diff_constant(time, trpl, n0)
    qfls = relative_QFLS(trpl, n0=n0, n0_max=1e18, eg=1.55)

    ax[1].semilogy(qfls, tau, color=color)
    ax_inset_1.loglog(time, tau, color=color)
    ax[2].semilogy(qfls, d, color=color)
    ax_inset_2.loglog(time, d, color=color)

ax[0].set_xlabel(time_label, fontsize='large')
ax[0].set_ylabel(PL_label, fontsize='large')
ax[0].set_title('TRPL Decay Curves')
ax[0].legend(title=fr'k$_{{e}}$ (ns$^{{-1}}$)', loc = 'lower left', fontsize='large', title_fontsize='large')
ax[1].set_xlabel(qfls_label, fontsize='large')
ax[1].set_ylabel(diff_decay_label, fontsize='large')
ax[1].set_title('Diff. Lifetime vs QFLS', fontsize='large')
ax_inset_1.set_xlabel(time_label, fontsize='medium')
ax_inset_2.set_xlabel(time_label, fontsize='medium')
ax[2].set_xlabel(qfls_label, fontsize='large')
ax[2].set_ylabel(diff_const_label, fontsize='large')
ax[2].set_title('Diff. Constant vs QFLS', fontsize='large')
ax[0].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[1].tick_params(axis='both', which='both', labelsize='large', direction='in')
ax[2].tick_params(axis='both', which='both', labelsize='large', direction='in')

fig.tight_layout()
k_e = 1e-4 #Reset to the default value